In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import pairwise_distances
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import copy

In [ ]:
artnet_2025= ** Tabular Data**

In [ ]:
artnet_2025_embed = ** Image Embedding**

In [ ]:
artnet_2025[artnet_2025["workyear from"] == 1998].index

In [ ]:
year_to_prior = {}
current_prior = set()
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Start Building Prior Knowledge")
for year in sorted(artnet_2025["workyear from"].unique()):
    year_rows_index = artnet_2025[artnet_2025["workyear from"] == year].index
    year_to_prior[year] = copy.deepcopy(current_prior)  # store snapshot before update
    current_prior |= set(year_rows_index)
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Finish Building Prior Knowledge")

In [ ]:
def compute_newness(i,row_dict, year_to_prior,embeddings,BASELINE_YEAR):
    year = row_dict["workyear from"]
    prior_knowledge = embeddings[list(year_to_prior.get(year, set()))]
    A = embeddings[i]
    if year <= BASELINE_YEAR:  # pre-year baseline
        return i, 0
    # print(f"The total steps are: {playcount}")
    # mark new pairs not in prior knowledge
    distances = np.linalg.norm(prior_knowledge - A, axis=1)
    min_dist = np.min(distances)
    return i, min_dist

In [ ]:
range_start = 200000
number_size = 100000
BASELINE_YEAR=1600
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [ ]:
newness = np.zeros(N)
max_workers = 10  # tune to your CPU cores
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Parallel computation starts")
with ThreadPoolExecutor(max_workers=max_workers) as ex:
    futures = {
        ex.submit(compute_newness, i, 
                artnet_2025.iloc[i],
                year_to_prior,
                artnet_2025_embed,
                BASELINE_YEAR): i
        for i in range(range_start,range_end)
    }
    
    completed = 0
    for fut in as_completed(futures):
        i,row_newness = fut.result()
        newness[i-range_start]  = row_newness
        completed += 1
        if completed % 10000 == 0:
            print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Completed {completed}/{artnet_2025.shape[0]}")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}: Ends")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Newness\\newness_{range_start}.npy", newness)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Embedding Saved")